In [2]:
import pandas as pd
import numpy as np

In [3]:
# ============================================================
# Configuration
# ============================================================
DATA_PATH = r"A:\Projects\DATA 202\Movies_and_TV_5.json.gz"

EXPECTED_COLUMNS = ["item_id", "user_id", "rating", "timestamp"]

# Optional expected rating bounds for Amazon-style ratings
EXPECTED_RATING_MIN = 1
EXPECTED_RATING_MAX = 5

# ============================================================
# Load data
# ============================================================
# This notebook supports either:
# 1. A cleaned CSV with columns: item_id, user_id, rating, timestamp
# 2. The original Amazon review file in .json.gz format
if str(DATA_PATH).lower().endswith((".json.gz", ".json")):
    df = pd.read_json(DATA_PATH, lines=True, compression="infer")

    # Map original Amazon columns to the project's working schema
    amazon_column_map = {
        "asin": "item_id",
        "reviewerID": "user_id",
        "overall": "rating",
        "unixReviewTime": "timestamp",
    }
    missing_amazon_cols = [col for col in amazon_column_map if col not in df.columns]
    if missing_amazon_cols:
        raise ValueError(f"Missing expected Amazon columns: {missing_amazon_cols}")

    df = df.rename(columns=amazon_column_map)
    df = df[EXPECTED_COLUMNS].copy()
else:
    df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Check required columns
missing_expected = [col for col in EXPECTED_COLUMNS if col not in df.columns]
if missing_expected:
    raise ValueError(f"Missing expected columns: {missing_expected}")

# Keep only the working columns in the expected order
df = df[EXPECTED_COLUMNS].copy()

# ============================================================
# Basic structure and types
# ============================================================
print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes)

# Coerce rating and timestamp to numeric if needed
df["rating_numeric"] = pd.to_numeric(df["rating"], errors="coerce")
df["timestamp_numeric"] = pd.to_numeric(df["timestamp"], errors="coerce")

# ============================================================
# Core dataset summary
# ============================================================
n_rows = len(df)
n_users = df["user_id"].nunique(dropna=True)
n_items = df["item_id"].nunique(dropna=True)

rating_min = df["rating_numeric"].min()
rating_max = df["rating_numeric"].max()

print("\n" + "=" * 60)
print("CORE SUMMARY")
print("=" * 60)
print(f"Row count: {n_rows}")
print(f"Unique users: {n_users}")
print(f"Unique items: {n_items}")
print(f"Minimum rating: {rating_min}")
print(f"Maximum rating: {rating_max}")

# ============================================================
# Missing values and duplicates
# ============================================================
missing_values = df[EXPECTED_COLUMNS].isna().sum()
duplicate_rows = df.duplicated().sum()

print("\n" + "=" * 60)
print("MISSING VALUES BY COLUMN")
print("=" * 60)
print(missing_values)

print("\n" + "=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)
print(f"Duplicate rows: {duplicate_rows}")

# ============================================================
# Rating distribution
# ============================================================
rating_distribution = (
    df["rating_numeric"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n" + "=" * 60)
print("RATING DISTRIBUTION COUNTS")
print("=" * 60)
print(rating_distribution)

# ============================================================
# Ratings per user
# ============================================================
ratings_per_user = df.groupby("user_id").size()
ratings_per_user_summary = ratings_per_user.describe()

print("\n" + "=" * 60)
print("RATINGS PER USER SUMMARY")
print("=" * 60)
print(ratings_per_user_summary)

print(f"\nAverage ratings per user: {ratings_per_user.mean():.4f}")
print(f"Median ratings per user: {ratings_per_user.median():.4f}")

# ============================================================
# Ratings per item
# ============================================================
ratings_per_item = df.groupby("item_id").size()
ratings_per_item_summary = ratings_per_item.describe()

print("\n" + "=" * 60)
print("RATINGS PER ITEM SUMMARY")
print("=" * 60)
print(ratings_per_item_summary)

print(f"\nAverage ratings per item: {ratings_per_item.mean():.4f}")
print(f"Median ratings per item: {ratings_per_item.median():.4f}")

# ============================================================
# Timestamp conversion
# ============================================================
df["datetime"] = pd.to_datetime(df["timestamp_numeric"], unit="s", errors="coerce")

min_date = df["datetime"].min()
max_date = df["datetime"].max()

print("\n" + "=" * 60)
print("TIMESTAMP SUMMARY")
print("=" * 60)
print(f"Earliest date: {min_date}")
print(f"Latest date: {max_date}")

# Optional calendar features
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month

year_counts = df["year"].value_counts(dropna=False).sort_index()
month_counts = df["month"].value_counts(dropna=False).sort_index()

print("\n" + "=" * 60)
print("YEAR COUNTS")
print("=" * 60)
print(year_counts)

print("\n" + "=" * 60)
print("MONTH COUNTS")
print("=" * 60)
print(month_counts)

# ============================================================
# Sparsity estimate
# ============================================================
observed_ratings = n_rows
possible_ratings = n_users * n_items

if possible_ratings > 0:
    sparsity = 1 - (observed_ratings / possible_ratings)
else:
    sparsity = np.nan

print("\n" + "=" * 60)
print("MATRIX SPARSITY")
print("=" * 60)
print(f"Observed ratings: {observed_ratings}")
print(f"Possible user-item pairs: {possible_ratings}")
print(f"Sparsity estimate: {sparsity:.6f}" if pd.notna(sparsity) else "Sparsity estimate: NaN")

# ============================================================
# Suspicious / impossible values
# ============================================================
invalid_rating_mask = (
    df["rating_numeric"].notna() &
    (
        (df["rating_numeric"] < EXPECTED_RATING_MIN) |
        (df["rating_numeric"] > EXPECTED_RATING_MAX)
    )
)

failed_timestamp_parse = df["datetime"].isna().sum()
nulls_in_key_columns = df[["item_id", "user_id", "rating", "timestamp"]].isna().sum()

print("\n" + "=" * 60)
print("DATA QUALITY FLAGS")
print("=" * 60)

if invalid_rating_mask.any():
    print(f"WARNING: Found {invalid_rating_mask.sum()} rating(s) outside the expected range "
          f"[{EXPECTED_RATING_MIN}, {EXPECTED_RATING_MAX}].")
else:
    print("OK: No ratings found outside the expected range.")

if failed_timestamp_parse > 0:
    print(f"WARNING: {failed_timestamp_parse} timestamp value(s) failed to parse into datetime.")
else:
    print("OK: All timestamps parsed successfully.")

if duplicate_rows > 0:
    print(f"WARNING: Found {duplicate_rows} duplicate row(s).")
else:
    print("OK: No duplicate rows found.")

key_null_total = nulls_in_key_columns.sum()
if key_null_total > 0:
    print("WARNING: Null values detected in key columns:")
    print(nulls_in_key_columns[nulls_in_key_columns > 0])
else:
    print("OK: No null values found in key columns.")

# ============================================================
# Additional concise outputs for slide prep
# ============================================================
print("\n" + "=" * 60)
print("SLIDE-READY SUMMARY")
print("=" * 60)
print(f"Rows: {n_rows}")
print(f"Unique users: {n_users}")
print(f"Unique items: {n_items}")
print(f"Rating range observed: {rating_min} to {rating_max}")
print(f"Duplicate rows: {duplicate_rows}")
print(f"Date range: {min_date} to {max_date}")
print(f"Average ratings per user: {ratings_per_user.mean():.4f}")
print(f"Average ratings per item: {ratings_per_item.mean():.4f}")
print(f"Sparsity estimate: {sparsity:.6f}" if pd.notna(sparsity) else "Sparsity estimate: NaN")


DATASET LOADED
Shape: (3410019, 4)
Columns: ['item_id', 'user_id', 'rating', 'timestamp']

DATA TYPES
item_id        str
user_id        str
rating       int64
timestamp    int64
dtype: object

CORE SUMMARY
Row count: 3410019
Unique users: 297529
Unique items: 60175
Minimum rating: 1
Maximum rating: 5

MISSING VALUES BY COLUMN
item_id      0
user_id      0
rating       0
timestamp    0
dtype: int64

DUPLICATE ROWS
Duplicate rows: 115340

RATING DISTRIBUTION COUNTS
rating_numeric
1     193169
2     172439
3     349700
4     665920
5    2028791
Name: count, dtype: int64

RATINGS PER USER SUMMARY
count    297529.000000
mean         11.461132
std          24.909173
min           1.000000
25%           5.000000
50%           7.000000
75%          11.000000
max        3509.000000
dtype: float64

Average ratings per user: 11.4611
Median ratings per user: 7.0000

RATINGS PER ITEM SUMMARY
count    60175.000000
mean        56.668367
std        179.928211
min          1.000000
25%          8.00000